# 分支管理

分支是 Git 最强大的特性。Git 的分支极其轻量——它只是一个指向某个提交的指针。这让你可以随时创建分支做实验，而不影响主线。

你将学到：
1. 理解分支的本质
2. 创建、切换、删除分支
3. 分支管理的最佳实践


---

## 1. 理解分支的本质

**核心原理：Git 保存的是一次次的快照（提交），分支只是指向某个快照的名字；创建分支不复制任何代码，在哪个分支上提交，那个名字就移到新的快照上。**

拆成三个要点：

1. **提交 = 快照**：每次 `git commit`，Git 把当时的全部文件“拍照”存档——就是上一课档案柜里的一本卷宗
2. **分支 = 指向某份快照的名字**：像贴在书页上的书签；程序员术语叫“指针”，意思就是“指向某次提交”
3. **名字跟着提交走**：你在哪个分支上提交，哪个分支的名字就滑到新快照上，其他分支不动

下面用你亲手提交过的三次记录，把这三点走一遍：

```
Initial commit: add README     ← 记作 C1
docs: 添加 Features 章节         ← C2
chore: 添加 .gitignore          ← C3（最新）
```

把三次提交画成一条线（从左到右越来越新）：

```
(C1) ── (C2) ── (C3)
                  ↑
                  main
```

**main 分支的全部内容，就是这样一张“书签”：它指在 C3 上**。所谓“main 上的代码”，就是 C3 记录的那份快照——此刻即“加了 .gitignore 的版本”。分支本身不带任何代码，它只是一个标记。

**创建分支 = 多放一张书签**

执行 `git branch feature` 后：

```
(C1) ── (C2) ── (C3)
                  ↑
                  ├── feature（新建的）
                  └── main
```

`main` 和 `feature` 都指在 **C3** 上——没有复制任何文件，两张书签指在同一处。这就是“创建分支零成本”的原因。

**Git 的书签与真书签唯一的区别：你一提交，它自动前移**

接着上面的场景（main、feature 都指在 C3），你在 feature 分支上提交了一次，产生新提交 C4：

```
(C1) ── (C2) ── (C3) ── (C4)   ← 刚刚提交产生的
                  ↑      ↑
                  │      └─ feature：C3 → C4，跟上了
                  └─ main：留在 C3，没动
```

发生了什么，一步步看：

1. 新提交 C4 记录你这次的改动，接在 C3 后面
2. feature 发现“这次提交是在我身上做的”，自动前移指到 C4
3. main 与这次提交无关，停在 C3 不动——所以“main 上的代码”仍是 C3 那份快照

规则很简单：**在哪个分支上提交，就移动哪个分支**。怎么从一个分支换到另一个分支？下一节马上讲。

>开分支 = 加一个名字，删分支 = 撕掉名字，都不碰任何提交和代码。所以放心大胆地开分支——新功能、实验、重构，都值得单独开一个分支去做；做得不满意，删掉分支，一切照旧。


---

## 2. 创建、切换、删除分支

### 2.1 准备练习仓库

用第 2 课学过的 init + add + commit，快速搭一个带一次提交的练习仓库：


In [ ]:
import os, shutil
demo = "/tmp/git_branch_demo"
if os.path.exists(demo):
    shutil.rmtree(demo)
os.makedirs(demo)
os.chdir(demo)

!git init
# 指定分支名为 main（Git 2.28+ 可直接用 git init -b main）
!git checkout -b main
!git config user.name "Demo User"
!git config user.email "demo@example.com"

# 创建初始提交
with open("app.py", "w") as f: f.write("print('v1.0')\n")
!git add app.py
!git commit -m "feat: 初始版本 v1.0"

# 查看当前分支
!git branch


### 2.2 创建和切换分支

两个核心命令，一"建"一"切"：

```bash
git branch <名字>       # 创建分支（人还留在当前分支）
git checkout <名字>     # 切换过去（人挪到新分支）
```

下面动手体验：先创建 `feature/login` 但**不切换**，看看分支列表；再切换过去，对比 `*` 的位置变化：


In [ ]:
# 创建一个新分支 feature/login（但不切换）
!git branch feature/login

# 查看所有分支：* 标记当前分支——此时仍在 main 上
!git branch -v

# 切换到新分支
!git checkout feature/login
# 或使用更新的命令（推荐）：
# git switch feature/login

# 切换后再看一次：* 移到了 feature/login
!git branch -v


### 2.3 一步到位：创建并切换

"先建后切"两步太常用，Git 提供了合并版——日常最常用的分支命令：

```bash
git checkout -b <名字>   # 创建并立即切换（= branch + checkout 两步合一）
```


In [ ]:
# 创建并切换到新分支（最常用！）
!git checkout -b feature/register
# 或新语法：
# git switch -c feature/register

!git branch

### 2.4 在分支上开发并提交

分支上的开发与平时完全一样：add → commit。区别只是提交只推进**当前分支**（§1 的规则）。提交后用图形化日志观察"名字"的移动：

```bash
git add <file>
git commit -m "..."              # 推进当前分支
git log --oneline --graph --all  # 图形化查看所有分支的位置
```

在 `feature/register` 分支上添加注册功能：


In [ ]:
# 在 feature/register 分支上添加注册功能
with open("app.py", "a") as f:
    f.write("def register():\n    print('register')\n")

!git add app.py
!git commit -m "feat(register): 添加注册功能"

# 查看历史：feature/register 比 main 多一个提交
!git log --oneline --graph --all

### 2.5 删除分支

分支只是名字，删除 = 撕掉名字——已合并进历史的提交一个不少：

```bash
git branch -d <名字>    # 删除已合并的分支（未合并会拒绝，安全）
git branch -D <名字>    # 强制删除（连未合并的提交一起丢，慎用）
```


In [ ]:
# 切回 main（不能删除当前所在分支）
!git checkout main

# 删除已合并的分支（安全）
!git branch -d feature/login

# 强制删除未合并的分支（危险！会丢失未合并的工作）
# git branch -D feature/register

!git branch

---

## 3. 分支管理的最佳实践

### 3.1 常见分支模型

| 分支 | 用途 |
|------|------|
| `main` / `master` | 生产分支，始终保持可发布状态 |
| `develop` | 开发集成分支，最新开发成果 |
| `feature/*` | 功能分支，从 develop 拉出，合并回 develop |
| `fix/*` | 修复分支 |
| `hotfix/*` | 紧急修复，从 main 拉出，合并回 main 和 develop |
| `release/*` | 发布准备分支 |

### 3.2 分支命名规范

```bash
feature/user-login       # 功能分支
fix/memory-leak          # bug 修复
hotfix/security-patch    # 紧急修复
docs/api-reference       # 文档
refactor/auth-module     # 重构
```

### 3.3 日常建议

1. **频繁合并**：不要让分支偏离主线太久，定期 `git merge main` 或 `git rebase main`
2. **小步提交**：每个分支只做一件事，完成后尽快合并
3. **及时清理**：合并后删除已合并的分支
4. **不要在 main 上直接开发**：始终通过功能分支
5. **推送前 rebase**（可选）：`git rebase main` 让历史保持线性

---

## 4. 总结

**回顾分支核心命令：**

```bash
git branch                    # 查看分支
git branch <name>             # 创建分支
git checkout -b <name>        # 创建并切换（最常用）
git checkout <name>           # 切换分支
git branch -d <name>          # 删除分支
git log --oneline --graph --all  # 查看分支图
```

接下来，请学习 [远程仓库协作](./04_git_remote.ipynb)，将本地仓库推送到远程！